# 02 — Multi-Factor Model & IC Analysis

Building a composite factor, computing IC, plotting quintile spreads.

In [ ]:
import pandas as pd
from hailmary.data.providers import YahooFinanceProvider
from hailmary.models.factors import MomentumFactor, VolatilityFactor
from hailmary.models.multi_factor import MultiFactorModel
from hailmary.analytics.statistics import FactorAnalytics
from hailmary.viz.factor_charts import FactorCharts

In [ ]:
# Fetch a broad universe
yahoo = YahooFinanceProvider()
universe = ['AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','JPM','V','UNH',
            'XOM','JNJ','PG','MA','HD','BAC','PFE','ABBV','LLY','MRK']
bars = yahoo.get_bars(universe, start='2019-01-01', end='2024-01-01')
close = bars['close'].unstack(level=0).dropna(how='all')

In [ ]:
# Build a momentum + low-vol multi-factor model
model = MultiFactorModel([
    MomentumFactor(lookback=252, skip=21),
    VolatilityFactor(window=63),
], combination='equal')

# Compute factor score panel (monthly rebalance)
score_panel = model.score_panel(close, frequency='ME')
print(score_panel.shape)
score_panel.tail()

In [ ]:
# Forward returns (21-day)
fwd_returns = close.pct_change(21).shift(-21)

# IC analysis
analytics = FactorAnalytics(score_panel, fwd_returns, n_quantiles=5)
print(analytics.ic_stats(horizon=21))

In [ ]:
# Visualise
charts = FactorCharts(analytics, factor_name='Momentum + Low-Vol')
charts.tearsheet(horizon=21).show()

In [ ]:
charts.ic_decay_chart(max_horizon=63).show()